# Ручная проверка API поиска

Ноутбук отправляет HTTP-запрос в локальный FastAPI и показывает результаты поиска. Сначала запустите сервис из корня проекта: `python -m uvicorn src.api.main:app --reload`. Swagger доступен по адресу `http://127.0.0.1:8000/docs`. Для получения результатов нужен утверждённый корпус `data/knowledge/articles.jsonl`; без него API вернёт 503 и продолжит показывать статус базы знаний.


In [ ]:
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

API_ROOT = 'http://127.0.0.1:8000'


In [ ]:
try:
    with urlopen(f'{API_ROOT}/health/live', timeout=5) as response:
        print('API:', json.loads(response.read().decode('utf-8')))
except URLError as error:
    print('Не удалось подключиться к API. Запустите uvicorn из корня проекта.')
    print(error)


In [ ]:
try:
    with urlopen(f'{API_ROOT}/health/knowledge/ready', timeout=5) as response:
        print('База знаний:', json.loads(response.read().decode('utf-8')))
except HTTPError as error:
    body = json.loads(error.read().decode('utf-8'))
    print(f'База знаний пока не готова: HTTP {error.code}')
    print(body)
except URLError as error:
    print('API недоступен. Сначала проверьте предыдущую ячейку.')
    print(error)


## Отправка поискового запроса

Измените запрос на свой. `top_k` ограничен API значением 10. Фильтр `language` можно оставить пустым или указать код языка статей.


In [ ]:
payload = {
    'text': 'card payment reversed',
    'top_k': 5,
    'language': 'en',
}
request = Request(
    f'{API_ROOT}/v1/knowledge/search',
    data=json.dumps(payload).encode('utf-8'),
    headers={'Content-Type': 'application/json'},
    method='POST',
)
try:
    with urlopen(request, timeout=15) as response:
        result = json.loads(response.read().decode('utf-8'))
    print('Request ID:', result['request_id'])
    print('Ранжировщик:', result['retriever'])
    print('Статус порога:', result['retrieval_status'])
    print('Порог для языка:', result['threshold'])
    for position, hit in enumerate(result['hits'], start=1):
        print(f"{position}. {hit['title']}")
        print(f"   Статья: {hit['document_id']} | фрагмент: {hit['chunk_id']}")
        print(f"   Источник: {hit['source']} | актуальность: {hit['updated_at']}")
        print(f"   Score: {hit['score']:.4f} ({hit['score_type']})")
        print(hit['text'])
except HTTPError as error:
    body = json.loads(error.read().decode('utf-8'))
    print(f'Поиск не выполнен: HTTP {error.code}')
    print(body)
except URLError as error:
    print('Не удалось подключиться к API.')
    print(error)


## Измерение полного HTTP-запроса

При запущенном API и готовом индексе эта ячейка отправляет до 100 calibration-запросов дважды и измеряет полный round trip до получения JSON-байтов. Она не сохраняет исходные тексты или тела ответов: только короткий SHA-256 идентификатор запроса, HTTP-статус и latency. Результаты попадают в локальный `work/api_runs/`, игнорируемый Git. Это ориентир на текущем компьютере, а не нагрузочный тест или SLO. Holdout не читается.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import statistics
import time
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
calibration_path = ROOT / 'data' / 'knowledge' / 'queries_calibration.jsonl'
if not calibration_path.is_file():
    print('Добавьте calibration-набор, чтобы выполнить измерение HTTP latency.')
else:
    from src.rag.evaluation import load_queries
    calibration_examples = load_queries(calibration_path)
    try:
        with urlopen(f'{API_ROOT}/health/knowledge/ready', timeout=5) as response:
            knowledge_health = json.loads(response.read().decode('utf-8'))
        if knowledge_health.get('status') != 'ready':
            raise RuntimeError('Knowledge API readiness не ready.')
        def timed_search(example):
            payload = {'text': example['query'], 'top_k': 5}
            if example.get('language'):
                payload['language'] = example['language']
            request = Request(
                f'{API_ROOT}/v1/knowledge/search',
                data=json.dumps(payload, ensure_ascii=False).encode('utf-8'),
                headers={'Content-Type': 'application/json'},
                method='POST',
            )
            started = time.perf_counter()
            try:
                with urlopen(request, timeout=15) as response:
                    response.read()
                    status_code = response.status
            except HTTPError as error:
                error.read()
                status_code = error.code
            except TimeoutError:
                status_code = 'timeout'
            except URLError:
                status_code = 'network_error'
            latency = (time.perf_counter() - started) * 1000
            query_id = hashlib.sha256(example['query'].encode('utf-8')).hexdigest()[:16]
            return {'query_id': query_id, 'status_code': status_code, 'latency_ms': latency}
        benchmark_examples = calibration_examples[:100]
        warmup_examples = benchmark_examples[:min(3, len(benchmark_examples))]
        for example in warmup_examples:
            timed_search(example)
        measurements = []
        for repeat_index in range(1, 3):
            for example in benchmark_examples:
                row = timed_search(example)
                row['repeat'] = repeat_index
                measurements.append(row)
        http_results = pd.DataFrame(measurements)
        successful = http_results[http_results['status_code'] == 200]['latency_ms']
        http_summary = {
            'run_id': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'api_root': API_ROOT,
            'knowledge_health': knowledge_health,
            'calibration_sha256': hashlib.sha256(calibration_path.read_bytes()).hexdigest(),
            'query_count': len(benchmark_examples),
            'repeat_count': 2,
            'request_count': len(http_results),
            'status_counts': {str(key): int(value) for key, value in http_results['status_code'].value_counts().items()},
            'successful_p50_ms': float(np.percentile(successful, 50)) if len(successful) else None,
            'successful_p95_ms': float(np.percentile(successful, 95)) if len(successful) else None,
            'successful_mean_ms': float(statistics.mean(successful)) if len(successful) else None,
        }
        run_dir = ROOT / 'work' / 'api_runs' / http_summary['run_id']
        run_dir.mkdir(parents=True, exist_ok=False)
        http_results.to_csv(run_dir / 'http_requests.csv', index=False)
        (run_dir / 'http_summary.json').write_text(json.dumps(http_summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        display(pd.DataFrame([http_summary]))
        display(http_results.groupby('status_code').agg(
            request_count=('latency_ms', 'size'),
            p50_ms=('latency_ms', 'median'),
            p95_ms=('latency_ms', lambda values: float(np.percentile(values, 95))),
        ))
        print(f'Локальные HTTP-замеры сохранены: {run_dir}')
    except (HTTPError, URLError, TimeoutError, RuntimeError) as error:
        print(f'HTTP benchmark пропущен: {error}')


## Как оценивать выдачу

Проверьте, ведёт ли `source` к актуальной одобренной статье и действительно ли фрагмент подтверждает нужный ответ. `score` пригоден для сортировки только внутри выбранного ранжировщика; это не confidence. Поле `retrieval_status` показывает, настроен ли порог и прошёл ли top-1 его проверку; сами кандидаты остаются видны для ручной оценки. Это не гарантия корректности. Ноутбук не генерирует ответ клиенту. Для оценки качества используйте размеченную выборку и notebook `02_retrieval_exploration.ipynb`.
